<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/SpectraProcessing_v8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpectraProcessing v8

Batch colorimetric processing of absorbance spectra from a user-uploaded ZIP archive. The notebook plots complete source spectra, calculates RGB, CIELAB, and Munsell values, displays a colour table, and exports a DOCX summary.

## Method and workflow

Run Cells 1–9 consecutively. Absorbance is multiplied by the common scaling factor before conversion with $T(\lambda)=10^{-A(\lambda)}$. Plots retain every valid source wavelength, while colour integration uses 360–780 nm and the CIE 1931 2° observer.

CIELAB uses the illuminant selected in Cell 3; display sRGB is adapted to D65; Munsell notation is calculated under Illuminant C. The interactive Plotly display always uses solid calculated-colour lines. Cell 8 independently selects one of three Matplotlib representations for the static figure embedded in DOCX: solid coloured lines, patterned coloured lines, or patterned black lines. Dot-only patterns are excluded.

In [ ]:
#@title 1. Check packages, import libraries, and set project metadata
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.specifiers import SpecifierSet
from packaging.version import Version

REQUIRED={
    "numpy":("numpy",">=1.24,<2.3"),
    "pandas":("pandas",">=2.0"),
    "scipy":("scipy",">=1.10,<1.15"),
    "colour":("colour-science",">=0.4.6,<0.5"),
    "plotly":("plotly",">=5.18"),
    "docx":("python-docx",">=1.1"),
    "matplotlib":("matplotlib",">=3.7"),
}
install=[]
for module,(distribution,specifier) in REQUIRED.items():
    needed=importlib.util.find_spec(module) is None
    if not needed:
        try:
            needed=Version(version(distribution)) not in SpecifierSet(specifier)
        except PackageNotFoundError:
            needed=True
    if needed:
        install.append(f"{distribution}{specifier}")
if install:
    print("Installing missing or incompatible packages:",", ".join(install))
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*install])
else:
    print("All required packages are installed at compatible versions.")

from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import html
import re
import unicodedata
import warnings
import zipfile
import numpy as np
import pandas as pd
import colour
import plotly.graph_objects as go
from IPython.display import display, HTML

VERSION="8"
OUTPUT_STEM=f"SpectraProcessing_v{VERSION}"
OBSERVER="CIE 1931 2 Degree Standard Observer"
SPECTRAL_SHAPE=colour.SpectralShape(360,780,1)
CMFS=colour.MSDS_CMFS[OBSERVER].copy().align(SPECTRAL_SHAPE)
SUPPORTED_EXTENSIONS={".txt",".csv",".dat",".asc"}
print(f"{OUTPUT_STEM} | colour-science {colour.__version__}")

All required packages are installed at compatible versions.
SpectraProcessing_v6 | colour-science 0.4.7


In [ ]:
#@title 2. Upload a ZIP archive of spectra
try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This upload cell is intended for Google Colab.") from exc

uploaded=files.upload()
zip_names=[name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names)!=1:
    raise ValueError(f"Select exactly one ZIP archive; received {list(uploaded)}")
zip_path=Path(zip_names[0])
zip_path.write_bytes(uploaded[zip_names[0]])
print(f"Archive ready: {zip_path.name} ({zip_path.stat().st_size:,} bytes)")

Saving pH_txt.zip to pH_txt (1).zip
Archive ready: pH_txt (1).zip (29,111 bytes)


In [ ]:
#@title 3. Choose calculation illuminant
illuminant = "D65" #@param ["A", "C", "D50", "D55", "D65", "D75", "E", "FL2", "FL7", "FL11"]
if illuminant not in colour.SDS_ILLUMINANTS:
    raise ValueError(f"Unavailable illuminant: {illuminant}")
selected_illuminant=str(illuminant)
print("Selected calculation illuminant:",selected_illuminant)

Selected calculation illuminant: D65


In [ ]:
#@title 4. Define parsing and colour-calculation functions
NUMBER_PATTERN=re.compile(r"[-+]?\d+(?:[.,]\d+)?(?:[Ee][-+]?\d+)?")


def decode_spectrum(raw):
    if raw.startswith((b"\xff\xfe",b"\xfe\xff")):
        return raw.decode("utf-16"),"utf-16"
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw.decode("utf-8-sig"),"utf-8-sig"
    for encoding in ("utf-8","cp1251","cp866","cp1252","latin-1"):
        try:
            text=raw.decode(encoding)
            if "\x00" not in text:
                return text,encoding
        except UnicodeDecodeError:
            pass
    raise UnicodeError("Spectrum text encoding could not be identified.")


def parse_spectrum(raw,name):
    text,encoding=decode_spectrum(raw)
    rows=[]
    candidate_lines=0
    for line_number,line in enumerate(text.splitlines(),1):
        tokens=NUMBER_PATTERN.findall(line)
        if len(tokens)<2:
            continue
        candidate_lines+=1
        try:
            wavelength=float(tokens[0].replace(",","."))
            absorbance=float(tokens[1].replace(",","."))
        except ValueError:
            continue
        if np.isfinite(wavelength) and np.isfinite(absorbance):
            rows.append((wavelength,absorbance,line_number))
    if len(rows)<2:
        preview=" | ".join(repr(line) for line in text.splitlines()[:6])
        raise ValueError(
            f"found {len(rows)} usable rows after {encoding} decoding; "
            f"candidate numeric lines={candidate_lines}; first lines: {preview}"
        )
    frame=pd.DataFrame(rows,columns=["wavelength_nm","absorbance","line"])
    frame=frame[(frame.wavelength_nm>=200)&(frame.wavelength_nm<=2500)]
    frame=(frame.groupby("wavelength_nm",as_index=False).absorbance
                .mean().sort_values("wavelength_nm"))
    if len(frame)<2:
        raise ValueError("fewer than two plausible spectral rows remained")
    lower,upper=frame.wavelength_nm.min(),frame.wavelength_nm.max()
    if lower>360 or upper<780:
        raise ValueError(f"coverage {lower:g}–{upper:g} nm does not span 360–780 nm")
    frame.attrs["encoding"]=encoding
    return frame


def filename_quality(text):
    cyrillic=sum("\u0400"<=char<="\u04ff" for char in text)
    boxes=sum("\u2500"<=char<="\u259f" for char in text)
    invalid=text.count("�")+sum(ord(char)<32 for char in text)
    mojibake=len(re.findall(r"[ÐÑРС][^\s]",text))
    return sum(char.isprintable() for char in text)+3*cyrillic-10*boxes-15*invalid-4*mojibake


def repair_filename(name):
    candidates=[name]
    for encoding in ("cp1251","latin-1","cp1252"):
        try:
            candidates.append(name.encode(encoding).decode("utf-8"))
        except UnicodeError:
            pass
    return max(candidates,key=filename_quality)


def choose_zip_encoding(path):
    candidates=[]
    for encoding in (None,"cp1251","cp866","cp437"):
        try:
            with zipfile.ZipFile(path,"r",metadata_encoding=encoding) as archive:
                score=sum(filename_quality(repair_filename(item.filename)) for item in archive.infolist())
                candidates.append((score,encoding))
        except (UnicodeError,zipfile.BadZipFile):
            pass
    if not candidates:
        raise zipfile.BadZipFile("ZIP archive or filename encoding could not be read.")
    return max(candidates,key=lambda item:item[0])[1]


def whitepoint_xy(name):
    return np.asarray(colour.CCS_ILLUMINANTS[OBSERVER][name],float)


def build_munsell_reference():
    labels=[]
    xyY=[]
    for (hue,value,chroma),coordinates in colour.notation.MUNSELL_COLOURS_ALL:
        labels.append(f"{hue} {value:g}/{chroma:g}")
        xyY.append([coordinates[0],coordinates[1],coordinates[2]*0.975/100])
    for value in np.arange(0.1,10.0,0.1):
        label=f"N{value:.1f}"
        labels.append(label)
        xyY.append(colour.munsell_colour_to_xyY(label))
    reference_Lab=colour.XYZ_to_Lab(
        colour.xyY_to_XYZ(np.asarray(xyY,float)),illuminant=whitepoint_xy("C")
    )
    return np.asarray(labels,object),reference_Lab


MUNSELL_LABELS,MUNSELL_LAB=build_munsell_reference()


def munsell_code(xyY):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if not colour.volume.is_within_macadam_limits(xyY,"C"):
                raise ValueError("outside MacAdam limits")
            return str(colour.xyY_to_munsell_colour(xyY)),"direct"
    except (ValueError,RuntimeError,FloatingPointError) as exc:
        Lab=colour.XYZ_to_Lab(
            colour.xyY_to_XYZ(xyY),illuminant=whitepoint_xy("C")
        )
        delta=colour.delta_E(MUNSELL_LAB,Lab,method="CIE 2000")
        index=int(np.nanargmin(delta))
        return str(MUNSELL_LABELS[index]),(
            f"nearest renotation; ΔE00={delta[index]:.2f}; direct: {type(exc).__name__}"
        )


def calculate_colour(frame,name,illuminant_name,scaling_factor):
    wavelength=frame.wavelength_nm.to_numpy(float)
    scaled_absorbance=frame.absorbance.to_numpy(float)*scaling_factor
    transmittance=np.clip(10**(-scaled_absorbance),0,1)
    sd=colour.SpectralDistribution(
        dict(zip(wavelength,transmittance)),name=name
    ).align(SPECTRAL_SHAPE)

    illuminant_sd=colour.SDS_ILLUMINANTS[illuminant_name].copy().align(SPECTRAL_SHAPE)
    XYZ=colour.sd_to_XYZ(sd,CMFS,illuminant_sd,method="Integration")/100
    source_white=whitepoint_xy(illuminant_name)
    Lab=colour.XYZ_to_Lab(XYZ,source_white)
    rgb=np.clip(colour.XYZ_to_sRGB(
        XYZ,illuminant=source_white,chromatic_adaptation_transform="Bradford"
    ),0,1)
    rgb255=np.rint(rgb*255).astype(int)

    illuminant_c=colour.SDS_ILLUMINANTS["C"].copy().align(SPECTRAL_SHAPE)
    XYZ_c=colour.sd_to_XYZ(sd,CMFS,illuminant_c,method="Integration")/100
    munsell,method=munsell_code(colour.XYZ_to_xyY(XYZ_c))

    return {
        "file":name,"caption":Path(name).stem,
        "wavelength":wavelength,"absorbance":scaled_absorbance,
        "source_range":(float(wavelength.min()),float(wavelength.max())),
        "Lab":np.asarray(Lab,float),"rgb":np.asarray(rgb,float),
        "rgb255":rgb255,"hex":"#{:02X}{:02X}{:02X}".format(*rgb255),
        "munsell":munsell,"munsell_method":method,
        "encoding":frame.attrs["encoding"],
    }

In [ ]:
#@title 5. Set scaling and plot appearance, then process spectra
scaling_factor = 1.0 #@param {type:"number"}
line_width = 2.0 #@param {type:"number"}
font_size = 14 #@param {type:"integer"}
font_family = "serif" #@param ["serif", "sans-serif", "monospace", "Arial", "Times New Roman"]

scaling_factor=float(scaling_factor)
line_width=float(line_width)
font_size=int(font_size)
font_family=str(font_family)
if scaling_factor<0:
    raise ValueError("Scaling factor must be non-negative.")
if line_width<=0 or font_size<6:
    raise ValueError("Line width must be positive and font size at least 6.")
if "zip_path" not in globals() or not Path(zip_path).exists():
    raise RuntimeError("Run Cell 2 before processing.")
if "selected_illuminant" not in globals():
    raise RuntimeError("Run Cell 3 before processing.")

metadata_encoding=choose_zip_encoding(zip_path)
spectra=[]
processing_errors=[]
with zipfile.ZipFile(zip_path,"r",metadata_encoding=metadata_encoding) as archive:
    members=[item for item in archive.infolist()
             if not item.is_dir()
             and not item.filename.startswith("__MACOSX/")
             and Path(item.filename).suffix.lower() in SUPPORTED_EXTENSIONS]
    if not members:
        raise ValueError("No supported spectrum files were found.")
    for member in members:
        name=repair_filename(PurePosixPath(member.filename).name)
        try:
            frame=parse_spectrum(archive.read(member),name)
            spectra.append(calculate_colour(
                frame,name,selected_illuminant,scaling_factor
            ))
        except Exception as exc:
            processing_errors.append({"file":name,"error":str(exc)})
if not spectra:
    raise RuntimeError(f"No spectra were processed: {processing_errors}")

# Sort all successfully processed spectra by filename.
# Natural sorting: 2-spectrum.txt precedes 10-spectrum.txt.

def filename_sort_key(spectrum):
    name = unicodedata.normalize("NFC", spectrum["file"]).casefold()
    parts = re.split(r"(\d+)", name)
    return tuple(
        (1, int(part)) if part.isdigit() else (0, part)
        for part in parts
    )

spectra.sort(key=filename_sort_key)

print(f"Processed {len(spectra)} spectrum/s; illuminant={selected_illuminant}; scaling={scaling_factor:g}")
for spectrum in spectra:
    low,high=spectrum["source_range"]
    print(f"  {spectrum['file']}: plotted {low:g}–{high:g} nm; "
          f"encoding={spectrum['encoding']}; Munsell={spectrum['munsell']}")
if processing_errors:
    display(pd.DataFrame(processing_errors))

Processed 18 spectrum/s; illuminant=D65; scaling=1
  pH2.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=9.1R 8.4/6.1
  pH3.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=9.6R 8.4/6.0
  pH3.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=0.2YR 8.6/5.4
  pH4.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=1.1YR 8.7/4.7
  pH4.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=2.1YR 8.6/4.8
  pH5.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=4.2Y 9.4/6.6
  pH5.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=9.1R 8.4/6.1
  pH6.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=3.0GY 9.1/3.7
  pH6.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=4.4GY 9.1/3.2
  pH7.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=7.7GY 8.8/2.4
  pH7.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=2.1G 8.6/2.1
  pH8.0.txt: plotted 300–800 nm; encoding=utf-8; Munsell=0.6B 8.0/2.5
  pH8.5.txt: plotted 300–800 nm; encoding=utf-8; Munsell=7.0BG 8.2/2.4
  pH9.0.txt: plotted 300–800 nm;

In [ ]:
#@title 6. Display complete source spectra with calculated colours
fig=go.Figure()
for spectrum in spectra:
    fig.add_trace(go.Scatter(
        x=spectrum["wavelength"],y=spectrum["absorbance"],mode="lines",
        name=spectrum["caption"],
        line={"color":spectrum["hex"],"width":line_width},
        hovertemplate=(f"<b>{html.escape(spectrum['caption'])}</b><br>"
                       "Wavelength: %{x:.1f} nm<br>Scaled absorbance: %{y:.4f}<extra></extra>")
    ))
axis_style={"showline":True,"linecolor":"black","linewidth":1,
            "mirror":True,"ticks":"outside","showgrid":False,"zeroline":False}
fig.update_layout(
    template="plotly_white",
    font={"family":font_family,"size":font_size,"color":"black"},
    xaxis={"title":"Wavelength (nm)",**axis_style},
    yaxis={"title":"Scaled absorbance",**axis_style},
    legend={"x":1.02,"xanchor":"left","y":1,"yanchor":"top"},
    margin={"l":75,"r":280,"t":35,"b":70},
    width=1100,height=620,plot_bgcolor="white",paper_bgcolor="white"
)
fig.show()

In [ ]:
#@title 7. Display the colour summary table
def triplet(values,decimals):
    return "("+", ".join(f"{float(value):.{decimals}f}" for value in values)+")"

summary_df=pd.DataFrame([{
    "File":spectrum["file"],
    "Colour":"",
    "RGB (0–255)":"("+", ".join(map(str,spectrum["rgb255"]))+")",
    "RGB (0–1)":triplet(spectrum["rgb"],4),
    "CIELAB (L*, a*, b*)":triplet(spectrum["Lab"],3),
    "Munsell (Illuminant C)":spectrum["munsell"],
    "Munsell assignment":spectrum["munsell_method"],
} for spectrum in spectra])

swatch_styles=[f"background-color:{spectrum['hex']};min-width:70px;height:24px"
               for spectrum in spectra]
styled=(summary_df.style
        .apply(lambda column:swatch_styles,subset=["Colour"])
        .set_properties(**{"text-align":"left","white-space":"nowrap"}))
display(styled)

,File,Colour,RGB (0–255),RGB (0–1),"CIELAB (L*, a*, b*)",Munsell (Illuminant C),Munsell assignment
0,pH2.5.txt,,"(255, 194, 173)","(1.0000, 0.7597, 0.6783)","(84.617, 23.997, 20.988)",9.1R 8.4/6.1,direct
1,pH3.0.txt,,"(255, 194, 172)","(1.0000, 0.7613, 0.6738)","(84.562, 23.213, 21.505)",9.6R 8.4/6.0,direct
2,pH3.5.txt,,"(255, 199, 177)","(1.0000, 0.7816, 0.6922)","(85.748, 20.720, 20.646)",0.2YR 8.6/5.4,direct
3,pH4.0.txt,,"(255, 205, 183)","(1.0000, 0.8052, 0.7169)","(87.038, 17.447, 19.084)",1.1YR 8.7/4.7,direct
4,pH4.5.txt,,"(255, 204, 178)","(1.0000, 0.8015, 0.6972)","(86.619, 16.947, 21.132)",2.1YR 8.6/4.8,direct
5,pH5.0.txt,,"(255, 236, 178)","(1.0000, 0.9266, 0.6964)","(93.901, -1.338, 31.095)",4.2Y 9.4/6.6,direct
6,pH5.5.txt,,"(255, 194, 173)","(1.0000, 0.7597, 0.6783)","(84.617, 23.997, 20.988)",9.1R 8.4/6.1,direct
7,pH6.0.txt,,"(230, 235, 184)","(0.9038, 0.9229, 0.7220)","(91.675, -9.907, 24.379)",3.0GY 9.1/3.7,direct
8,pH6.5.txt,,"(226, 234, 187)","(0.8882, 0.9187, 0.7323)","(91.154, -10.429, 22.275)",4.4GY 9.1/3.2,direct
9,pH7.0.txt,,"(209, 228, 194)","(0.8210, 0.8946, 0.7589)","(88.460, -12.359, 14.809)",7.7GY 8.8/2.4,direct


In [ ]:
#@title 8. Choose static plot representation for DOCX export
static_plot_representation = "Solid coloured lines" #@param ["Solid coloured lines", "Patterned coloured lines", "Patterned black lines"]

STATIC_PLOT_OPTIONS={
    "Solid coloured lines",
    "Patterned coloured lines",
    "Patterned black lines",
}
if static_plot_representation not in STATIC_PLOT_OPTIONS:
    raise ValueError(f"Unknown static plot representation: {static_plot_representation}")
selected_static_representation=str(static_plot_representation)
print("Static DOCX plot representation:",selected_static_representation)

Static DOCX plot representation: Solid coloured lines


In [ ]:
#@title 9. Export the selected static plot and table to DOCX using Matplotlib
import matplotlib.pyplot as plt
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt

DASH_LENGTH={"dot":1.0,"short":3.5,"long":8.0}


def ternary_pattern(number,width):
    digits=[]
    for _ in range(width):
        digits.append(number%3)
        number//=3
    names=("dot","short","long")
    return tuple(names[digit] for digit in reversed(digits))


def make_patterned_styles(count):
    if count<1:
        return []
    width=1
    while 3**width-1<count:
        width+=1
    styles=[]
    code=0
    while len(styles)<count:
        pattern=ternary_pattern(code,width)
        code+=1
        if all(element=="dot" for element in pattern):
            continue
        sequence=[]
        for element in pattern:
            sequence.extend((DASH_LENGTH[element],2.2))
        sequence[-1]=5.5
        styles.append({"name":"–".join(pattern),"linestyle":(0,tuple(sequence))})
    signatures=[repr(style["linestyle"]) for style in styles]
    if len(signatures)!=len(set(signatures)):
        raise RuntimeError("Repeated static dash pattern was generated.")
    if any(set(style["name"].split("–"))=={"dot"} for style in styles):
        raise RuntimeError("A prohibited dot-only pattern was generated.")
    return styles


def set_cell_fill(cell,colour_value):
    properties=cell._tc.get_or_add_tcPr()
    node=properties.find(qn("w:shd"))
    if node is None:
        node=OxmlElement("w:shd")
        properties.append(node)
    node.set(qn("w:fill"),colour_value.lstrip("#"))


def make_static_plot(path):
    patterned=selected_static_representation!="Solid coloured lines"
    styles=make_patterned_styles(len(spectra)) if patterned else None
    with plt.rc_context({
        "font.family":font_family,"font.size":font_size,
        "axes.labelsize":font_size,"legend.fontsize":max(6,font_size-1),
        "lines.dash_capstyle":"butt",
    }):
        figure,axes=plt.subplots(figsize=(12,6.8),dpi=220)
        artists=[]
        for index,spectrum in enumerate(spectra):
            colour_value=("black" if selected_static_representation=="Patterned black lines"
                          else spectrum["hex"])
            linestyle=styles[index]["linestyle"] if patterned else "-"
            artist,=axes.plot(
                spectrum["wavelength"],spectrum["absorbance"],
                color=colour_value,linewidth=line_width,linestyle=linestyle,
                label=spectrum["caption"],
            )
            artists.append(artist)
        if patterned:
            normalized=[repr(getattr(artist,"_unscaled_dash_pattern",None))
                        for artist in artists]
            if len(normalized)!=len(set(normalized)):
                raise RuntimeError("Matplotlib normalized two patterns identically.")
            print("Verified unique patterns (no dot-only pattern):",
                  "; ".join(style["name"] for style in styles))
        axes.set_xlabel("Wavelength (nm)")
        axes.set_ylabel("Scaled absorbance")
        axes.grid(False)
        axes.tick_params(direction="out",top=False,right=False)
        for spine in axes.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1)
        axes.legend(loc="upper left",bbox_to_anchor=(1.01,1),
                    frameon=False,handlelength=4.5)
        figure.tight_layout()
        figure.savefig(path,bbox_inches="tight",facecolor="white")
        plt.close(figure)


if "selected_static_representation" not in globals():
    raise RuntimeError("Run Cell 8 before exporting.")
output_docx=Path(f"{OUTPUT_STEM}_summary.docx")
output_png=Path(f"{OUTPUT_STEM}_spectra.png")
make_static_plot(output_png)

document=Document()
section=document.sections[0]
section.orientation=WD_ORIENT.LANDSCAPE
section.page_width,section.page_height=section.page_height,section.page_width
for attribute in ("left_margin","right_margin","top_margin","bottom_margin"):
    setattr(section,attribute,Inches(0.55))
document.styles["Normal"].font.name="Arial"
document.styles["Normal"].font.size=Pt(8)

paragraph=document.add_heading(f"{OUTPUT_STEM}: Spectral colour summary",0)
paragraph.alignment=WD_ALIGN_PARAGRAPH.CENTER
paragraph=document.add_paragraph(
    f"Illuminant: {selected_illuminant} | Scaling factor: {scaling_factor:g} | "
    f"Static representation: {selected_static_representation} | CIE 1931 2° | "
    f"{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}"
)
paragraph.alignment=WD_ALIGN_PARAGRAPH.CENTER

document.add_heading("Complete source spectra",1)
document.add_picture(str(output_png),width=Inches(9.7))
document.paragraphs[-1].alignment=WD_ALIGN_PARAGRAPH.CENTER

document.add_heading("Colour coordinates",1)
table=document.add_table(rows=1,cols=len(summary_df.columns))
table.style="Table Grid"
for column,name in enumerate(summary_df.columns):
    table.rows[0].cells[column].text=name
    set_cell_fill(table.rows[0].cells[column],"D9E2F3")
for row_index,(_,row) in enumerate(summary_df.iterrows()):
    cells=table.add_row().cells
    for column,name in enumerate(summary_df.columns):
        cells[column].text="" if name=="Colour" else str(row[name])
    set_cell_fill(cells[list(summary_df.columns).index("Colour")],
                  spectra[row_index]["hex"])

if processing_errors:
    document.add_heading("Skipped files",1)
    for error in processing_errors:
        document.add_paragraph(f"{error['file']}: {error['error']}",style="List Bullet")

document.add_heading("Method note",1)
document.add_paragraph(
    "All valid source wavelengths were plotted; colour integration used 360–780 nm. "
    f"The static figure representation was '{selected_static_representation}'. "
    "Patterned modes use unique ordered combinations of dot, short-dash, and long-dash "
    "elements; dot-only patterns are excluded. The static figure was rendered with "
    "Matplotlib, without Kaleido."
)
document.save(output_docx)
print("DOCX created:",output_docx.resolve())

try:
    from google.colab import files
    files.download(str(output_docx))
except ImportError:
    display(HTML(f'<a href="{output_docx}" download>Download {output_docx.name}</a>'))

DOCX created: /content/SpectraProcessing_v6_summary.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>